# Assignment: Pricing an American Put with Q-Learning

You'll build a **tabular Q-learning** agent that learns the optimal exercise policy for an American put option, by training directly on a binomial tree -- then check its price and exercise boundary against that same tree's exact, independently computed backward-induction solution.

Cells marked **TODO** are yours to complete. Everything else is given. Don't modify `market_sim.py`.

## The option to price

An American **put**:

| Parameter | Value |
|---|---|
| Spot $S_0$ | \$100 |
| Strike $K$ | \$100 |
| Risk-free rate $r$ | 6% |
| Volatility $\sigma$ | 30% |
| Maturity $T$ | 1 year |

Priced on a 25-step CRR binomial tree, as in the option lecture that precedes this assignment.

*Notation:* in this notebook $r$ is the risk-free **interest rate** (not the reward) and `p` is the tree's **up-probability** (not the look-ahead steps of the TD lecture); the reward is just "the reward".

## Part 1 -- Formulate the MDP (no code)

Before writing any code, write out the MDP for *this specific option*, using the general state/action/reward/discount framework from the RL lectures (A0-A4), now on the tree:

- What is the state $s$? (Think in terms of $(t, j)$, not $(t, S_t)$.)
- What are the actions?
- What is the reward for each action, including at maturity?
- What is $\gamma$, numerically, for one tree step? (The code calls it `disc`.)

Answer in the cell below.

**Your answer:**

*(replace this cell)*

## Part 2 -- The tree environment (given)

`market_sim.py` provides these functions (the first two appear in the option lecture's code cells):

- `binomial_tree_params(r, sigma, T, n_steps)` -- returns `(u, d, p, disc)`: the up/down price *factors* (multipliers -- $u > 1$ means an up-move **raises** the price, $d < 1$ means a down-move **lowers** it), the risk-neutral up-probability, and the one-step discount factor $e^{-r\Delta t}$.
- `node_price(S0, u, d, t, j)` -- the price at tree node $(t,j)$, where $j$ is the number of down-moves so far ($0 \le j \le t$): $S_0 u^{\,t-j} d^{\,j}$. Since only the *count* of down-moves matters (not their order -- the tree recombines), there are exactly $t+1$ distinct reachable prices at time $t$, one per value of $j$. **Larger $j$ means *lower* price** -- keep that in mind when you extract the exercise boundary in Part 5.
- `binomial_tree_american_put(S0, K, r, sigma, T, n_steps)` -- returns `(price, boundary)`, the exact price and exercise boundary via backward induction on this same tree. **Don't look at this until Part 5** -- it's your answer key, not a shortcut.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from market_sim import binomial_tree_params, node_price, binomial_tree_american_put

S0, K, r, sigma, T = 100.0, 100.0, 0.06, 0.30, 1.0
n_steps = 25
u, d, p, disc = binomial_tree_params(r, sigma, T, n_steps)

CONTINUE, EXERCISE = 0, 1
n_actions = 2

def payoff(S):
    return np.maximum(K - S, 0.0)

## Part 3 -- Warm-up: where does a plain walk from the root go? (TODO)

Simulate a few random walks through the tree, always starting at the root $(0, 0)$: at each step, the price moves **up** (multiplied by `u`, price rises; `j` stays the same) with probability `p`, else **down** (multiplied by `d`, price falls; `j += 1`).

Plot ~10 such walks, price vs. time step, on the same axes. (Create a generator with `rng = np.random.default_rng(0)`; `rng.random()` returns a uniform number in $[0, 1)$.)

Then answer: **at a late time step (say $t=20$), do your walks reach nodes with very high $j$ (many down-moves) as often as nodes near the middle?** (Hint: think about how many *distinct* up/down sequences lead to each node.) You will meet this again in Part 6.

In [ ]:
N_WALKS = 10

# TODO: simulate N_WALKS random walks from (0, 0) to maturity and plot them


**Your observation:**

*(replace this cell)*

## Part 4 -- Implement Q-learning, starting every episode at the root (TODO)

You will build this in two stages. **Stage 1 (this part and Part 5): every training episode starts at the root $(0, 0)$.** Stage 2 (Part 6) changes only where episodes start.

Recall the Q-learning (TD) update from the A4 lecture:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\big[\text{target} - Q(s,a)\big], \qquad \text{target} = \text{reward} + \gamma \max_{a'}Q(s',a')$$

(here $\gamma$ is `disc` in the code).

Represent $Q$ as an array of shape `(n_steps + 1, n_steps + 1, n_actions)` -- one row per `(t, j)` (only entries with `j <= t` are ever used).

The training loop is given. Each episode gets its start `(t, j)` from a function `start_fn(rng)` and walks forward. At each step, at state `(t, j)`:

- Pick an action $\epsilon$-greedily from `Q[t, j]`.
- If **exercise**: `target = payoff(node_price(S0, u, d, t, j))` (no bootstrapping -- the episode ends here). Update `Q[t, j, EXERCISE]`; the loop then stops.
- If **continue**: reward is 0 this step. First **sample the actual transition**: with probability `p` the price moves up and the next node is `(t+1, j)` (same `j`, price rises), else it moves down and the next node is `(t+1, j+1)` (price falls) -- call its index `j_next`. This is important: even though you happen to know `p` here (it's given by the tree), *don't* use it to average over both branches in the target. That would be a different, model-based algorithm (a full expectation backup) -- the point of this exercise is the model-free Q-learning from A4, which bootstraps off the *one* node it actually transitions to, same as if `p` were unknown.
- The target at maturity and otherwise is already written in the loop. Update `Q[t, j, CONTINUE]`.

Because this is genuine sample-based learning, expect your price to land within a few percent of the tree's, not match it to three decimal places -- that residual gap is the real, honest signature of sample-based (TD) learning, not a bug to chase away. (Stage 1 will be worse than that; Part 5 shows how and Part 6 fixes it.)

**On the learning rate:** `alpha=0.01` is given, and so are `N_EPISODES=1_000_000` and `seed=0` -- keep all three fixed for the runs you submit; they were tuned together, and changing one without the others is exactly the kind of thing to try in Extra Credit, not in your graded runs. (A million episodes can look huge for a tree with only a few hundred states -- if you're curious exactly why that many are needed, see the option lecture's optional "deep dive" section at the very end.)

Briefly, here's why a **small constant** `alpha` is the right choice and not just an arbitrary one. A per-state decaying schedule like `alpha = 1 / N(s,a)` looks like the theoretically "safe" choice (it's what a running sample average uses) -- but Q-learning *bootstraps*: your target for one state depends on the current estimate of another state that's still being learned. A state visited early and often (like nodes near $t=0$) can shrink its per-state $\alpha$ to near zero before the states it bootstraps from have converged, freezing it at a stale value it can never correct. Going too far the other way is just as broken: `alpha` near 1 means each update *replaces* the estimate instead of blending with it, so the table just chases whatever single noisy sample arrived last. A small constant `alpha` sits between these two failure modes. (And `alpha = 0` is neither a safe nor a gentle choice -- it discards every update and learns nothing at all.)

In [ ]:
def epsilon_greedy_action(Q_row, epsilon, rng):
    '''Q_row: Q[t, j], shape (n_actions,). Return 0 or 1.'''
    # TODO: with probability epsilon, return a random action;
    # otherwise return the greedy action (np.argmax(Q_row))
    raise NotImplementedError


def root_start(rng):
    '''Stage 1: every episode starts at the root.'''
    return 0, 0


def train_q_learning(n_episodes, start_fn, alpha=0.01, epsilon_start=1.0,
                     epsilon_end=0.02, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((n_steps + 1, n_steps + 1, n_actions))

    for i in range(n_episodes):
        epsilon = epsilon_start + (epsilon_end - epsilon_start) * i / n_episodes
        t, j = start_fn(rng)

        while t < n_steps:
            S = node_price(S0, u, d, t, j)
            a = epsilon_greedy_action(Q[t, j], epsilon, rng)

            if a == EXERCISE:
                target = payoff(S)
                # TODO: TD-update Q[t, j, EXERCISE] toward target
                break
            else:
                # TODO: sample the actual transition -- with probability p,
                # j_next = j (up); otherwise j_next = j + 1 (down)
                j_next = ...

                if t + 1 == n_steps:
                    target = disc * payoff(node_price(S0, u, d, t + 1, j_next))
                else:
                    target = disc * np.max(Q[t + 1, j_next])
                # TODO: TD-update Q[t, j, CONTINUE] toward target

                j = j_next
                t += 1

    return Q


N_EPISODES = 1_000_000                 # given, don't change -- takes ~15-20 s
TRAIN_CONFIG = dict(alpha=0.01, seed=0)  # given, don't change
Q_root = train_q_learning(N_EPISODES, root_start, **TRAIN_CONFIG)  # Stage 1 run


## Part 5 -- Price the option and check the exercise boundary (TODO)

By the definition of $Q$, the option's value at $t=0$ is $\max_a Q(0, 0, a)$ (there's only one node at $t=0$: $j=0$).

The learned **policy** is $\arg\max_a Q(t, j, a)$ for every `(t, j)` with `j <= t`. The **exercise boundary** at time $t$ is the highest price whose node's policy says exercise. Since higher $j$ means *lower* price, that's the price at the **smallest** exercised $j$ at each $t$ -- not the largest.

Write `extract_results(Q)`, which returns the price, the policy and the boundary for any trained table. Then run the given comparison on your Stage 1 table `Q_root` (you will reuse the same function in Part 6).

Details that trip people up:

- `Q` is trained only for `t < n_steps` (at maturity the exercise is forced and no table entry is learned), so compare boundaries for `t = 0, ..., n_steps - 1`; the exact tree's value at `t = n_steps` has no counterpart in your table.
- Only entries with `j <= t` mean anything. Set `policy[t, j] = 0` for `j > t` (they are never used).
- If `Q[t, j, CONTINUE] == Q[t, j, EXERCISE]`, treat it as **continue** (`np.argmax` does this for you).
- If no node at time `t` says exercise, the boundary at `t` is `np.nan`.
- `policy` has shape `(n_steps + 1, n_steps + 1)` and `q_boundary` has length `n_steps + 1`, like `tree_boundary`; leave the entry of `q_boundary` at `t = n_steps` as `np.nan` so the two lines share an x-axis (`policy` is zero there).
- Learned values are noisy, so the exercised nodes at a given `t` may not be contiguous; the boundary is still the price at the smallest exercised `j`.

Expect Stage 1 to look clearly worse than the few-percent gap promised in Part 4 (Stage 2 typically ends within about 1% to 4% of the tree's price, with boundary misses of a single level; the summary reports both exact matches and matches within one level). Say in Part 7 which nodes you think are to blame.

In [ ]:
def extract_results(Q):
    '''Return (q_price, policy, q_boundary) for a trained Q table.'''
    # TODO: q_price = value at the root, max over actions of Q[0, 0]
    q_price = ...

    # TODO: policy[t, j] = 0 (continue) or 1 (exercise), for j <= t
    policy = ...

    # TODO: q_boundary[t] = node_price at the SMALLEST exercised j (highest price
    # still in the exercise region) among j in [0, t], else nan
    q_boundary = ...
    return q_price, policy, q_boundary


tree_price, tree_boundary = binomial_tree_american_put(S0, K, r, sigma, T, n_steps)


def compare_to_tree(Q, label):
    '''Given: prints the price comparison and draws the boundary plot.'''
    q_price, policy, q_boundary = extract_results(Q)
    print(f'[{label}] Q-learning price: {q_price:.3f}   Binomial tree price: {tree_price:.3f}')
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(q_boundary, label='Q-learning boundary')
    ax.plot(tree_boundary, label='Binomial tree boundary')
    ax.set_title(label)
    ax.set_xlabel('time step')
    ax.set_ylabel('stock price')
    ax.legend()
    plt.show()


compare_to_tree(Q_root, 'Stage 1: root starts')


## Part 6 -- Stage 2: exploring starts (TODO)

Part 3 showed that walks from the root rarely reach extreme nodes, especially at later time steps -- exactly the nodes the agent needs good data about. Recall **exploring starts** from the A4 lecture: generate training episodes starting from a node drawn broadly across the whole tree, not just the root.

Implement a function that:

1. Picks a random start time `t0` (an integer in `[0, n_steps-1]`).
2. Picks a random start node `j0` (an integer in `[0, t0]`).
3. Returns `(t0, j0)`.

(You do not change the training loop: it already takes the start function as an argument. Then rerun training with your function and compare with Stage 1.)

In [ ]:
def sample_exploring_start(rng):
    # TODO: t0 = a random integer in [0, n_steps - 1]
    t0 = ...
    # TODO: j0 = a random integer in [0, t0]
    j0 = ...
    return t0, j0


Q_explore = train_q_learning(N_EPISODES, sample_exploring_start, **TRAIN_CONFIG)  # Stage 2 run, given
compare_to_tree(Q_explore, 'Stage 2: exploring starts')


## Part 7 -- Reflection

Answer using the two required runs above -- don't change `n_steps`, `alpha`, `N_EPISODES`, or `seed` for these questions; that experimentation is what Extra Credit is for.

1. Report your Stage 1 price, your Stage 2 price, and the tree price.
2. Does your Stage 2 exercise boundary match the binomial tree's? Where (if anywhere) do they disagree, and what's your best guess why?
3. Stage 1 was worse. Which nodes do you think it learned badly, and why? (Connect this to Part 3.)
4. *(No code required)* Suppose this were a Bermudan option -- exercisable only every 5th time step, not every step. How would you change the action space or state representation to model that?
5. *(No code required)* You've now solved the *same* tree two ways: `binomial_tree_american_put` (exact backward induction, given) and your own Q-learning agent. The tree has only ~350 nodes, and backward induction solves it exactly in one pass -- yet your Q-learning agent needed a million *episodes* to get within a few percent. In your own words, what does Q-learning have to do without that backward induction gets for free, and why does that force it to be iterative and sample-based instead of a single sweep? (The tree has no cycles, so one backward pass is exact there.)

**Your answers:**

*(replace this cell)*

## Extra Credit -- break it on purpose

The fixed values in Part 4 (`n_steps`, `alpha`, `N_EPISODES`, `seed`) aren't arbitrary -- each was chosen to dodge a specific failure mode. These questions ask you to hit those failure modes deliberately and explain why, using the concepts from the lectures and Part 4. Do each as a separate experiment (copy the training cell rather than overwriting your required runs above).

1. Add visit counters to a copy of `train_q_learning` and count how often each node `(t, j)` is visited under `root_start` and under `sample_exploring_start`. Show both as heat maps (or tables) and connect them to the difference between your Stage 1 and Stage 2 prices.
2. Replace the constant `alpha` with a per-state decaying schedule: track a visit count `N[t, j, a]` and use `alpha = 1 / N[t, j, a]` for that update, where you **increment** `N[t, j, a]` by 1 *first* and then compute `alpha` (so the first update uses `alpha = 1`). Retrain with exploring starts and report the price. Using the bootstrapping argument from Part 4, explain why this theoretically-motivated choice converges to a *worse* answer here.
3. Set `alpha = 0.9` (still constant, just large) and retrain with a couple of different `seed`s (and rerun `alpha=0.01` with the same seeds for comparison). How much does the price vary from run to run compared to `alpha=0.01`? Connect this to the option lecture's explanation of why $\alpha$ near 1 never lets the table settle.

   Then, on a copy of the training cell, replace the sampled continue-target by the probability-weighted average of the two successors, `disc * (p * best_up + (1 - p) * best_down)`, and set `alpha = 1`. What price do you get, and why is this *not* Q-learning? Why does the tree's own computation not suffer the problem you saw with a sampled target at large `alpha`?
4. Try `n_steps = 10` and `n_steps = 100` (rebuilding `u, d, p, disc` and retraining for each, with the same `N_EPISODES`). How do the price and boundary change? Connect this to the resolution/data tradeoff from the option lecture.

   For each `n_steps`, rebind the globals `n_steps`, `u`, `d`, `p`, `disc` before calling `train_q_learning` (it reads them as globals), and rebuild your boundary arrays at the new size. On an otherwise idle machine, expect roughly 20 to 30 seconds even at 100 steps.
5. Does more training close the gap to the tree? Try `N_EPISODES = 4_000_000` (a minute or two) with `alpha=0.01`, and with `alpha=0.003`. Compare with your 1M-episode Stage 2 price and explain what you see.

**Your extra-credit answers (if attempted):**

*(replace this cell)*

## Submission summary (required)

This section is **required**: graders read it first. Run the notebook top to bottom, then run the cell below (it only reads what you already defined; don't edit it). Then fill in the written report. Fill in every field.

The summary cell prints, in a fixed format: your configuration, checks of your two TODO functions, the price and boundary results for both stages, and the source code of your functions.

In [ ]:
import inspect

GIVEN = dict(n_steps=25, alpha=0.01, seed=0, N_EPISODES=1_000_000)
USED = dict(n_steps=n_steps, alpha=TRAIN_CONFIG['alpha'], seed=TRAIN_CONFIG['seed'], N_EPISODES=N_EPISODES)
_sig = inspect.signature(train_q_learning).parameters
USED.update(epsilon_start=_sig['epsilon_start'].default, epsilon_end=_sig['epsilon_end'].default)
GIVEN.update(epsilon_start=1.0, epsilon_end=0.02)
changed = [k for k in GIVEN if USED[k] != GIVEN[k]]

print('=== SUBMISSION SUMMARY (printed automatically) ===')
print('config:', ', '.join(f'{k}={v}' for k, v in USED.items()))
print('given values unchanged:', 'YES' if not changed else 'NO -> ' + ', '.join(changed))

# --- check of epsilon_greedy_action: Q_row = [0, 1], so the greedy action is 1
_rng = np.random.default_rng(123)
_row = np.array([0.0, 1.0])
for _eps in (0.0, 0.5, 1.0):
    _frac = np.mean([epsilon_greedy_action(_row, _eps, _rng) == 1 for _ in range(20000)])
    print(f'epsilon_greedy_action: epsilon={_eps}: fraction greedy action = {_frac:.3f}')

# --- check of sample_exploring_start
_rng = np.random.default_rng(123)
_draws = [sample_exploring_start(_rng) for _ in range(20000)]
_t0 = np.array([d_[0] for d_ in _draws]); _j0 = np.array([d_[1] for d_ in _draws])
print(f'sample_exploring_start: t0 min/max = {_t0.min()}/{_t0.max()}, j0 > t0 count = {int((_j0 > _t0).sum())}, '
      f'j0 == t0 seen = {bool((_j0 == _t0).any())}, distinct nodes = {len(set(_draws))}')

# --- results of both stages
def _level_gap(a, b):
    '''Distance in price levels (one level = a factor u/d between neighbouring nodes); nan against a value counts as 99.'''
    if np.isnan(a) and np.isnan(b): return 0
    if np.isnan(a) or np.isnan(b): return 99
    return int(round(abs(np.log(a / b)) / np.log(u / d)))

def _boundary_matches(qb):
    gaps = [_level_gap(qb[t], tree_boundary[t]) for t in range(n_steps)]
    return (sum(g == 0 for g in gaps), sum(g <= 1 for g in gaps),
            [t for t, g in enumerate(gaps) if g > 0], max(gaps))

for _label, _Q in (('root starts', Q_root), ('exploring starts', Q_explore)):
    _price, _policy, _qb = extract_results(_Q)
    _n_ok, _n_one, _miss, _worst = _boundary_matches(_qb)
    print(f'results [{_label}]: price {_price:.3f}, tree {tree_price:.3f}, '
          f'difference {_price - tree_price:+.3f} ({(_price - tree_price) / tree_price:+.1%}); '
          f'boundary exact {_n_ok}/{n_steps}, within one level {_n_one}/{n_steps} (worst miss {_worst} levels; misses at t = {_miss}); '
          f'policy shape {np.shape(_policy)}, nonzero entries with j>t: '
          f'{int(sum(_policy[t][j] != 0 for t in range(n_steps + 1) for j in range(t + 1, n_steps + 1)))}, '
          f'boundary length {len(_qb)}, nan count {int(np.isnan(_qb).sum())}')

# --- the source of your functions, exactly as run
for _fn in (epsilon_greedy_action, train_q_learning, extract_results, sample_exploring_start):
    print('-' * 60)
    try:
        print(inspect.getsource(_fn))
    except (OSError, TypeError):
        print(f'(source of {_fn.__name__} not available -- paste it into the report below)')
print('=== END OF AUTOMATIC SUMMARY ===')


In [ ]:
# Written report -- fill in every field (a short answer is fine).
submission_summary = {
    "j_next_line": "",
    # Paste your exact line that samples j_next.

    "target_lines_edited": "",
    # Did you change the given `target = ...` lines in the training loop? If
    # yes, paste the new continue-target line and say which next node(s) it uses.
    # If no, write "no".

    "tie_rule": "",
    # In extract_results, what does your policy do when the two action values
    # are equal?

    "anything_changed_from_given": "",
    # Did you change any given code or value (including the given training
    # call)? If yes, what and why. If no, write "nothing".

    "stage1_vs_stage2": "",
    # One or two sentences: how do the Stage 1 and Stage 2 results differ?

    "extra_credit_attempted": "",
    # Which Extra Credit items did you attempt (e.g. "1, 3")? Write "none" if none.
}

for key, value in submission_summary.items():
    print(f"{key}:\n  {value}\n")


## What to submit

This notebook, fully run top to bottom (including the submission summary), with:
- All TODO cells completed, using the given `n_steps`, `alpha`, `N_EPISODES`, and `seed`
- The Stage 1 and Stage 2 comparisons from Parts 5 and 6
- Your written answers to Parts 1, 3, and 7, and the written report in the submission summary
- (Optional) Extra Credit answers

**Grading:** the assignment is graded out of **200 required points** (everything asked above, Parts 1-7), plus **up to 100 additional points** for going above and beyond (the Extra Credit items and extensions of your own).